In [1]:
# app.py
# Humanitarian Warehouse Compliance & Training Simulator
# Portfolio project – practical tool for CARE-style WIM capacity building & audit readiness

import streamlit as st
import pandas as pd
import random
import json
from datetime import datetime

# ────────────────────────────────────────────────
# Page Config & Dark Theme
# ────────────────────────────────────────────────
st.set_page_config(
    page_title="Warehouse Compliance & Training Simulator",
    page_icon="✅",
    layout="wide",
    initial_sidebar_state="expanded"
)

st.markdown("""
    <style>
        [data-testid="stAppViewContainer"] { background-color: #0e1117; color: #e0e0e0; }
        .stApp { background-color: #0e1117; }
        h1, h2, h3, h4, p, div, label { color: #f0f4f8 !important; }
        .sidebar .sidebar-content { background-color: #161b22; }
        .stMetric { background-color: #1f2937; border: 1px solid #374151; border-radius: 8px; padding: 1rem; }
        .stMetric label { color: #9ca3af !important; }
        .stMetric .stMetric-value { color: #60a5fa !important; }
        .stButton > button { background-color: #3b82f6; color: white; }
        .stButton > button:hover { background-color: #2563eb; }
        .footer-text { position: fixed; bottom: 0; left: 0; right: 0; background: #0e1117; text-align: center; padding: 12px; font-size: 0.85rem; color: #9ca3af; border-top: 1px solid #30363d; }
    </style>
""", unsafe_allow_html=True)

# ────────────────────────────────────────────────
# Title & Intro
# ────────────────────────────────────────────────
st.title("✅ Warehouse Compliance & Training Simulator")
st.markdown("Interactive tool for humanitarian warehousing staff to practice CARE Emergency Toolkit standards, conduct self-audits, identify risks, and receive corrective guidance.")

# ────────────────────────────────────────────────
# Load Data
# ────────────────────────────────────────────────
@st.cache_data
def load_data():
    df_scenarios = pd.read_csv("warehouse_scenarios.csv")
    df_checklist = pd.read_csv("compliance_checklist.csv")
    return df_scenarios, df_checklist

df_scenarios, df_checklist = load_data()

# ────────────────────────────────────────────────
# Core Functions (from Phase 2 & 3)
# ────────────────────────────────────────────────
def calculate_compliance_score(answers_dict):
    earned = 0.0
    possible = 0.0
    for _, row in df_checklist.iterrows():
        chk_id = row["check_id"]
        if chk_id not in answers_dict: continue
        answer = str(answers_dict[chk_id]).strip().title()
        if answer == "N/A": continue
        possible += row["weight"]
        if answer == "Yes": earned += row["weight"]
        elif answer == "Partial": earned += row["weight"] * 0.5
    percentage = round((earned / possible) * 100, 1) if possible > 0 else 0.0
    return {"percentage": percentage, "earned": earned, "possible": possible}

def get_risk_category(pct):
    if pct >= 90: return "Low Risk", "green"
    if pct >= 70: return "Medium Risk", "orange"
    if pct >= 50: return "High Risk", "red"
    return "Critical Risk", "darkred"

def generate_findings(answers_dict):
    findings = []
    for _, row in df_checklist.iterrows():
        chk_id = row["check_id"]
        if chk_id not in answers_dict: continue
        answer = str(answers_dict[chk_id]).strip().title()
        if answer in ["No", "Partial"]:
            action = "Immediately address. Review and update SOPs."
            if "expired" in row["requirement"].lower(): action += " Move to quarantine and document loss."
            if "fifo" in row["requirement"].lower() or "fefo" in row["requirement"].lower(): action += " Implement FIFO/FEFO system."
            if "ledger" in row["requirement"].lower() or "bin card" in row["requirement"].lower(): action += " Conduct full stock count."
            findings.append({
                "requirement": row["requirement"],
                "answer": answer,
                "weight": row["weight"],
                "action": action
            })
    return sorted(findings, key=lambda x: -x["weight"])

# ────────────────────────────────────────────────
# Sidebar – Scenario Selection
# ────────────────────────────────────────────────
with st.sidebar:
    st.header("Select Scenario")
    scenario_options = df_scenarios["title"].tolist()
    selected_title = st.selectbox("Warehouse Scenario", options=scenario_options)
    selected_scenario = df_scenarios[df_scenarios["title"] == selected_title].iloc[0]

    st.markdown("---")
    st.caption("Choose a realistic crisis context to begin training or audit simulation.")

# ────────────────────────────────────────────────
# Main Content – Two Modes
# ────────────────────────────────────────────────
tab_training, tab_audit = st.tabs(["📚 Training Mode", "🔍 Audit Simulation"])

# ── Training Mode ──────────────────────────────────────────────
with tab_training:
    st.subheader("Training Mode – Learn & Practice")
    st.markdown(selected_scenario["description"])
    st.caption(f"Key challenges: {selected_scenario['key_challenges']}")

    if st.button("Start Training Session", type="primary"):
        st.session_state["mode"] = "training"
        st.session_state["scenario"] = selected_scenario
        st.session_state["step"] = 0
        st.session_state["answers"] = {}
        st.session_state["questions"] = df_checklist.sample(frac=1).reset_index(drop=True)  # shuffle
        st.rerun()

# ── Audit Mode ─────────────────────────────────────────────────
with tab_audit:
    st.subheader("Audit Simulation Mode")
    st.markdown("Answer honestly as if conducting a real warehouse audit.")
    st.markdown(selected_scenario["description"])

    if st.button("Start Audit Simulation", type="primary"):
        st.session_state["mode"] = "audit"
        st.session_state["scenario"] = selected_scenario
        st.session_state["step"] = 0
        st.session_state["answers"] = {}
        st.session_state["questions"] = df_checklist.copy()
        st.rerun()

# ────────────────────────────────────────────────
# Interactive Quiz / Audit Flow
# ────────────────────────────────────────────────
if "step" in st.session_state and st.session_state["step"] < len(st.session_state["questions"]):
    mode = st.session_state["mode"]
    q_row = st.session_state["questions"].iloc[st.session_state["step"]]
    progress = (st.session_state["step"] + 1) / len(st.session_state["questions"])

    st.progress(progress)
    st.caption(f"Question {st.session_state['step']+1} of {len(st.session_state['questions'])}")

    st.markdown(f"**Category**: {q_row['category']}")
    st.markdown(f"**Requirement**: {q_row['requirement']}")
    st.caption(f"Evidence to check: {q_row['evidence']}")

    answer = st.radio(
        "Your assessment",
        options=["Yes", "Partial", "No", "N/A"],
        key=f"q_{q_row['check_id']}",
        horizontal=True
    )

    if st.button("Submit Answer & Continue"):
        st.session_state["answers"][q_row["check_id"]] = answer
        st.session_state["step"] += 1
        st.rerun()

elif "step" in st.session_state and st.session_state["step"] >= len(st.session_state["questions"]):
    # Final results
    answers = st.session_state["answers"]
    score = calculate_compliance_score(answers)
    risk_level, risk_color = get_risk_category(score["percentage"])
    findings = generate_findings(answers)

    st.success("Session Complete!")
    st.subheader(f"Compliance Score: {score['percentage']}%")
    st.metric("Risk Level", risk_level, delta=None)

    st.markdown(f"**Points earned**: {score['earned']} / {score['possible']}")

    if findings:
        st.subheader("Prioritized Findings & Recommended Actions")
        for i, f in enumerate(findings, 1):
            with st.expander(f"{i}. {f['requirement']} ({f['answer']}) – Weight {f['weight']}"):
                st.markdown(f"**Action**: {f['action']}")
    else:
        st.success("No major findings – excellent compliance!")

    # Export
    result_data = {
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "scenario": st.session_state["scenario"]["title"],
        "mode": st.session_state["mode"],
        "compliance_percentage": score["percentage"],
        "risk_level": risk_level,
        "findings": findings
    }

    st.download_button(
        "⬇️ Download Session Report (JSON)",
        json.dumps(result_data, indent=2),
        file_name=f"wim_{st.session_state['mode']}_{datetime.now().strftime('%Y%m%d_%H%M')}.json",
        mime="application/json"
    )

    if st.button("Start New Session"):
        for key in list(st.session_state.keys()):
            del st.session_state[key]
        st.rerun()

# Footer
st.markdown(
    "<div class='footer-text'>Humanitarian Warehouse Compliance Simulator • Built by Aklilu Abera</div>",
    unsafe_allow_html=True
)

2026-03-09 17:12:34.239 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-09 17:12:34.241 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-09 17:12:36.971 
  command:

    streamlit run C:\Users\Eldu\AppData\Roaming\Python\Python312\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-03-09 17:12:36.972 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-09 17:12:36.974 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-09 17:12:36.975 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-09 17:12:36.978 Thread 'MainThread': missing ScriptRunContext! This warning can be

DeltaGenerator()